# Обновленный код с обучением 

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
import os
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

# U-net архитектура

In [4]:
class DoubleConv(nn.Module):
    """(Conv3x3 -> BN -> ReLU) x 2"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    """Downscaling with maxpool then double conv"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    """Upscaling then double conv"""
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()

        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        # Pad x1 to the size of x2
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=1, bilinear=True):
        super(UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = OutConv(64, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits

# Работа с датасетом

In [5]:
class GridDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None, mask_transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.images = [f for f in os.listdir(image_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
        self.transform = transform
        self.mask_transform = mask_transform
        
        print(f"📂 Найдено изображений: {len(self.images)}")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        # Загружаем изображение
        img_name = self.images[idx]
        img_path = os.path.join(self.image_dir, img_name)
        
        # Ищем маску (может быть с разными расширениями)
        mask_name = img_name.replace('.jpg', '_mask.png').replace('.jpeg', '_mask.png').replace('.png', '_mask.png')
        mask_path = os.path.join(self.mask_dir, mask_name)
        
        # Если маски нет, пробуем другие варианты
        if not os.path.exists(mask_path):
            mask_name = img_name.replace('.jpg', '.png').replace('.jpeg', '.png')
            mask_path = os.path.join(self.mask_dir, mask_name)
        
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        if mask is None:
            raise FileNotFoundError(f"❌ Маска не найдена: {mask_path}")
        
        # Бинаризуем маску
        mask = (mask > 128).astype(np.float32)
        
        # Применяем трансформации
        if self.transform:
            image = self.transform(image)
        
        if self.mask_transform:
            mask = self.mask_transform(mask)
        else:
            mask = torch.from_numpy(mask).unsqueeze(0).float()
        
        return image, mask

# Метрики и визуализация результатов

In [6]:
def dice_coeff(pred, target, smooth=1.0):
    """Dice coefficient для оценки качества сегментации"""
    pred = torch.sigmoid(pred)
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    intersection = (pred_flat * target_flat).sum()
    return (2. * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth)

def visualize_prediction(model, dataset, device, num_samples=4):
    """Визуализация предсказаний модели"""
    model.eval()
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, num_samples*3))
    
    with torch.no_grad():
        for i in range(num_samples):
            idx = np.random.randint(len(dataset))
            image, mask = dataset[idx]
            image_batch = image.unsqueeze(0).to(device)
            
            pred = model(image_batch)
            pred = torch.sigmoid(pred).cpu().squeeze().numpy()
            
            # Денормализация изображения
            img_np = image.cpu().numpy().transpose(1, 2, 0)
            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            img_np = std * img_np + mean
            img_np = np.clip(img_np, 0, 1)
            
            axes[i, 0].imshow(img_np)
            axes[i, 0].set_title('Оригинал')
            axes[i, 0].axis('off')
            
            axes[i, 1].imshow(mask.cpu().squeeze(), cmap='gray')
            axes[i, 1].set_title('Маска (истина)')
            axes[i, 1].axis('off')
            
            axes[i, 2].imshow(pred, cmap='gray')
            axes[i, 2].set_title('Предсказание')
            axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

# Обучение

In [7]:
def train_grid_detector(image_dir='schemes', mask_dir='masks', epochs=50, batch_size=4):
    """Основная функция обучения"""
    
    # Проверяем наличие папок
    if not os.path.exists(image_dir):
        os.makedirs(image_dir)
        print(f"✅ Создана папка {image_dir}. Положи туда изображения схем!")
    
    if not os.path.exists(mask_dir):
        os.makedirs(mask_dir)
        print(f"✅ Создана папка {mask_dir}. Положи туда маски линий!")
    
    # Устройство
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"🚀 Используется устройство: {device}")
    
    # Трансформации
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    
    mask_transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
    ])
    
    # Датасет и загрузчик
    dataset = GridDataset(image_dir, mask_dir, transform=transform, mask_transform=mask_transform)
    
    if len(dataset) == 0:
        print("❌ Нет изображений в папке! Добавь изображения и маски.")
        return
    
    # Разделение на train/val
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    print(f"📊 Train: {len(train_dataset)} изображений, Val: {len(val_dataset)} изображений")
    
    # Модель
    model = UNet(n_channels=3, n_classes=1).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5)
    
    # Обучение
    best_val_loss = float('inf')
    train_losses = []
    val_losses = []
    
    print("\n🔨 НАЧАЛО ОБУЧЕНИЯ")
    print("="*50)
    
    for epoch in range(epochs):
        # Train
        model.train()
        train_loss = 0
        train_dice = 0
        
        for images, masks in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]'):
            images, masks = images.to(device), masks.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            train_dice += dice_coeff(outputs, masks).item()
        
        avg_train_loss = train_loss / len(train_loader)
        avg_train_dice = train_dice / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # Validation
        model.eval()
        val_loss = 0
        val_dice = 0
        
        with torch.no_grad():
            for images, masks in tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Val]'):
                images, masks = images.to(device), masks.to(device)
                outputs = model(images)
                loss = criterion(outputs, masks)
                
                val_loss += loss.item()
                val_dice += dice_coeff(outputs, masks).item()
        
        avg_val_loss = val_loss / len(val_loader)
        avg_val_dice = val_dice / len(val_loader)
        val_losses.append(avg_val_loss)
        
        scheduler.step(avg_val_loss)
        
        print(f"\n📊 Epoch {epoch+1}/{epochs}")
        print(f"   Train Loss: {avg_train_loss:.4f}, Train Dice: {avg_train_dice:.4f}")
        print(f"   Val Loss: {avg_val_loss:.4f}, Val Dice: {avg_val_dice:.4f}")
        print(f"   LR: {optimizer.param_groups[0]['lr']:.6f}")
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_grid_detector.pth')
            print(f"   💾 Лучшая модель сохранена! (Loss: {best_val_loss:.4f})")
        
        # Визуализация каждые 10 эпох
        if (epoch + 1) % 10 == 0:
            visualize_prediction(model, val_dataset, device)
    
    # Финальная визуализация
    print("\n✅ Обучение завершено!")
    visualize_prediction(model, val_dataset, device, num_samples=8)
    
    # Графики обучения
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(train_losses, label='Train Loss')
    plt.yscale('log')
    plt.xlabel('Epoch')
    plt.ylabel('Loss (log)')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig('training_history.png')
    plt.show()
    
    return model

# Использование обученной модели

In [8]:
def predict_grid(model, image_path, device='cpu'):
    """Предсказание сетки на новом изображении"""
    model.eval()
    
    # Загружаем и подготавливаем изображение
    image = cv2.imread(image_path)
    original_size = image.shape[:2]
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    
    input_tensor = transform(image_rgb).unsqueeze(0).to(device)
    
    # Предсказание
    with torch.no_grad():
        output = model(input_tensor)
        output = torch.sigmoid(output).cpu().squeeze().numpy()
    
    # Возвращаем к оригинальному размеру
    output = cv2.resize(output, (original_size[1], original_size[0]))
    output = (output > 0.5).astype(np.uint8) * 255
    
    return output

In [ ]:
if __name__ == "__main__":
    print("="*60)
    print("🔍 ОБУЧЕНИЕ ДЕТЕКТОРА СЕТКИ ДЛЯ СХЕМ ВЫШИВКИ")
    print("="*60)
    
    # Пути к папкам
    IMAGE_DIR = 'schemes'      # папка с изображениями схем
    MASK_DIR = 'masks'         # папка с масками линий сетки
    
    # Запуск обучения
    model = train_grid_detector(
        image_dir=IMAGE_DIR,
        mask_dir=MASK_DIR,
        epochs=50,
        batch_size=4
    )
    
    print("\n✅ Готово! Модель сохранена как 'best_grid_detector.pth'")

🔍 ОБУЧЕНИЕ ДЕТЕКТОРА СЕТКИ ДЛЯ СХЕМ ВЫШИВКИ
🚀 Используется устройство: cpu
📂 Найдено изображений: 0
❌ Нет изображений в папке! Добавь изображения и маски.

✅ Готово! Модель сохранена как 'best_grid_detector.pth'
